Install Libraries

In [9]:
!pip install -q fastapi uvicorn pypdf python-multipart \
    sentence-transformers faiss-cpu transformers sentencepiece \
    accelerate reportlab httpx

Import Libraries

In [10]:
import io
import time

import numpy as np
import pandas as pd

import faiss

from pypdf import PdfReader
from reportlab.pdfgen import canvas

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
from fastapi.testclient import TestClient

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("All libraries imported successfully.")

All libraries imported successfully.


Configuration

In [11]:
MAX_FILE_SIZE = 5 * 1024 * 1024   # 5 MB

CHUNK_SIZE = 100
CHUNK_OVERLAP = 20
TOP_K = 2

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "google/flan-t5-small"

print("Configuration loaded.")
print("Maximum file size:", MAX_FILE_SIZE, "bytes")
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Top K:", TOP_K)

Configuration loaded.
Maximum file size: 5242880 bytes
Chunk size: 100
Chunk overlap: 20
Top K: 2


Load Embedding Model

In [12]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


Load LLM

In [13]:
llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    LLM_MODEL
)

print("LLM loaded successfully.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully.


PDF Text Extraction

In [14]:
def extract_pdf_pages(file_bytes):
    try:
        reader = PdfReader(io.BytesIO(file_bytes))
    except Exception:
        raise ValueError("Invalid PDF file.")

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        pages.append({
            "page": page_number,
            "text": text.strip()
        })

    return pages

Chunking with Metadata

In [15]:
def create_chunks(
    pages,
    filename,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
):
    chunks = []
    metadata = []

    step = chunk_size - overlap

    for page_data in pages:
        page_number = page_data["page"]
        text = page_data["text"]

        if not text:
            continue

        words = text.split()

        for start in range(0, len(words), step):

            chunk_words = words[start:start + chunk_size]

            if not chunk_words:
                continue

            chunks.append(
                " ".join(chunk_words)
            )

            metadata.append({
                "filename": filename,
                "page": page_number,
                "chunk_id": len(chunks) - 1
            })

            if start + chunk_size >= len(words):
                break

    return chunks, metadata

Build FAISS Index

In [16]:
def build_faiss_index(chunks):

    if not chunks:
        raise ValueError(
            "No readable text found in the document."
        )

    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True
    ).astype("float32")

    index = faiss.IndexFlatL2(
        embeddings.shape[1]
    )

    index.add(embeddings)

    return index, embeddings

Create FastAPI Application

In [17]:
app = FastAPI(
    title="Day 17 RAG API",
    description="PDF upload and question answering using RAG",
    version="1.0"
)

print("FastAPI application created.")

FastAPI application created.


RAG Storage

In [18]:
rag_store = {
    "index": None,
    "chunks": [],
    "metadata": [],
    "filename": None,
    "pages": 0
}

print("RAG storage initialized.")

RAG storage initialized.


Question Request Model

In [19]:
class QuestionRequest(BaseModel):
    question: str
    top_k: int = TOP_K

Root and Health Endpoints

In [20]:
@app.get("/")
def root():
    return {
        "message": "Day 17 RAG API is running",
        "status": "healthy"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "document_loaded": rag_store["index"] is not None,
        "filename": rag_store["filename"]
    }

Upload Endpoint

In [21]:
@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...)):

    filename = file.filename or ""

    # Check file type
    if not filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format. Only PDF files are allowed."
        )

    # Read file
    file_bytes = await file.read()

    # Empty file check
    if not file_bytes:
        raise HTTPException(
            status_code=400,
            detail="Uploaded file is empty."
        )

    # File size check
    if len(file_bytes) > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=413,
            detail="File is too large. Maximum allowed size is 5 MB."
        )

    # Extract text
    try:
        pages = extract_pdf_pages(file_bytes)
    except ValueError as e:
        raise HTTPException(
            status_code=400,
            detail=str(e)
        )

    # Create chunks
    chunks, metadata = create_chunks(
        pages,
        filename
    )

    if not chunks:
        raise HTTPException(
            status_code=400,
            detail="PDF contains no extractable text."
        )

    # Create embeddings and FAISS index
    try:
        index, embeddings = build_faiss_index(
            chunks
        )
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Indexing failed: {str(e)}"
        )

    # Save current document
    rag_store["index"] = index
    rag_store["chunks"] = chunks
    rag_store["metadata"] = metadata
    rag_store["filename"] = filename
    rag_store["pages"] = len(pages)

    return {
        "message": "PDF uploaded and indexed successfully",
        "filename": filename,
        "pages": len(pages),
        "chunks": len(chunks),
        "embedding_dimension": embeddings.shape[1]
    }

Semantic Retrieval

In [22]:
def retrieve_chunks(
    query,
    top_k=TOP_K
):

    if rag_store["index"] is None:
        raise ValueError(
            "No PDF has been uploaded."
        )

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    top_k = min(
        top_k,
        rag_store["index"].ntotal
    )

    distances, indices = rag_store["index"].search(
        query_embedding,
        top_k
    )

    results = []

    for distance, index_value in zip(
        distances[0],
        indices[0]
    ):

        results.append({
            "chunk": rag_store["chunks"][index_value],
            "metadata": rag_store["metadata"][index_value],
            "distance": float(distance)
        })

    return results

Generate RAG Answer

In [23]:
def generate_answer(
    question,
    retrieved_chunks
):

    context = "\n\n".join(
        [
            f"Page {item['metadata']['page']}:\n"
            f"{item['chunk']}"
            for item in retrieved_chunks
        ]
    )

    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    answer = llm_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

Ask Endpoint

In [24]:
@app.post("/ask")
def ask_question(request: QuestionRequest):

    question = request.question.strip()

    if not question:
        raise HTTPException(
            status_code=400,
            detail="Question cannot be empty."
        )

    if rag_store["index"] is None:
        raise HTTPException(
            status_code=400,
            detail="Please upload a PDF first."
        )

    if request.top_k < 1:
        raise HTTPException(
            status_code=400,
            detail="top_k must be at least 1."
        )

    retrieved = retrieve_chunks(
        question,
        request.top_k
    )

    answer = generate_answer(
        question,
        retrieved
    )

    sources = []

    for item in retrieved:

        sources.append({
            "filename": item["metadata"]["filename"],
            "page": item["metadata"]["page"],
            "chunk_id": item["metadata"]["chunk_id"],
            "distance": round(
                item["distance"],
                4
            )
        })

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

Create Test Client

In [25]:
client = TestClient(app)

print("FastAPI test client created.")

FastAPI test client created.


Test Health API

In [26]:
response = client.get("/health")

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

Status Code: 200
Response:
{'status': 'healthy', 'document_loaded': False, 'filename': None}


Generate Sample PDF

In [26]:
def create_test_pdf():

    buffer = io.BytesIO()

    pdf = canvas.Canvas(buffer)

    pdf.setFont("Helvetica", 12)

    lines = [
        "NovaTech Solutions",
        "",
        "Customer Support",
        "Customer support is available Monday to Friday from 9 AM to 6 PM IST.",
        "Standard support requests are answered within 24 business hours.",
        "Priority support requests are targeted within 4 business hours.",
        "",
        "Product Plans",
        "The Starter plan costs $29 per month.",
        "The Professional plan costs $79 per month.",
        "The Professional plan supports up to 25 users.",
        "",
        "Leave Policy",
        "Full-time employees receive 18 paid leave days per calendar year.",
        "",
        "Security Policy",
        "Employees must use multi-factor authentication.",
        "Company credentials must never be shared.",
        "Security incidents must be reported immediately.",
        "",
        "Internship Program",
        "NovaTech internships normally run for four to six months."
    ]

    y = 800

    for line in lines:

        pdf.drawString(
            50,
            y,
            line
        )

        y -= 20

    pdf.save()

    buffer.seek(0)

    return buffer.getvalue()


sample_pdf = create_test_pdf()

print("Sample PDF created.")
print(
    "PDF size:",
    round(len(sample_pdf) / 1024, 2),
    "KB"
)

Upload PDF

In [28]:
def create_test_pdf():

    buffer = io.BytesIO()

    pdf = canvas.Canvas(buffer)

    pdf.setFont("Helvetica", 12)

    lines = [
        "NovaTech Solutions",
        "",
        "Customer Support",
        "Customer support is available Monday to Friday from 9 AM to 6 PM IST.",
        "Standard support requests are answered within 24 business hours.",
        "Priority support requests are targeted within 4 business hours.",
        "",
        "Product Plans",
        "The Starter plan costs $29 per month.",
        "The Professional plan costs $79 per month.",
        "The Professional plan supports up to 25 users.",
        "",
        "Leave Policy",
        "Full-time employees receive 18 paid leave days per calendar year.",
        "",
        "Security Policy",
        "Employees must use multi-factor authentication.",
        "Company credentials must never be shared.",
        "Security incidents must be reported immediately.",
        "",
        "Internship Program",
        "NovaTech internships normally run for four to six months."
    ]

    y = 800

    for line in lines:

        pdf.drawString(
            50,
            y,
            line
        )

        y -= 20

    pdf.save()

    buffer.seek(0)

    return buffer.getvalue()

sample_pdf = create_test_pdf()

start_time = time.time()

response = client.post(
    "/upload",
    files={
        "file": (
            "novatech.pdf",
            sample_pdf,
            "application/pdf"
        )
    }
)

upload_time = time.time() - start_time

print("Status Code:", response.status_code)
print("Upload Time:", round(upload_time, 2), "seconds")
print("Response:")
print(response.json())

Status Code: 200
Upload Time: 0.75 seconds
Response:
{'message': 'PDF uploaded and indexed successfully', 'filename': 'novatech.pdf', 'pages': 1, 'chunks': 2, 'embedding_dimension': 384}


Ask a Question

In [29]:
response = client.post(
    "/ask",
    json={
        "question": "What are the working hours for customer support?",
        "top_k": 2
    }
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

Status Code: 200
Response:
{'question': 'What are the working hours for customer support?', 'answer': 'Monday to Friday', 'sources': [{'filename': 'novatech.pdf', 'page': 1, 'chunk_id': 0, 'distance': 1.2185}, {'filename': 'novatech.pdf', 'page': 1, 'chunk_id': 1, 'distance': 1.6139}]}


Empty Question

In [30]:
response = client.post(
    "/ask",
    json={
        "question": "",
        "top_k": 2
    }
)

print("EMPTY QUESTION TEST")
print("Status Code:", response.status_code)
print("Response:", response.json())

EMPTY QUESTION TEST
Status Code: 400
Response: {'detail': 'Question cannot be empty.'}


Unsupported TXT File

In [31]:
response = client.post(
    "/upload",
    files={
        "file": (
            "document.txt",
            b"This is a text file.",
            "text/plain"
        )
    }
)

print("TXT FILE TEST")
print("Status Code:", response.status_code)
print("Response:", response.json())

TXT FILE TEST
Status Code: 400
Response: {'detail': 'Unsupported file format. Only PDF files are allowed.'}


Unsupported DOCX File

In [32]:
response = client.post(
    "/upload",
    files={
        "file": (
            "document.docx",
            b"This is a DOCX file.",
            "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
        )
    }
)

print("DOCX FILE TEST")
print("Status Code:", response.status_code)
print("Response:", response.json())

DOCX FILE TEST
Status Code: 400
Response: {'detail': 'Unsupported file format. Only PDF files are allowed.'}


Empty PDF File

In [33]:
response = client.post(
    "/upload",
    files={
        "file": (
            "empty.pdf",
            b"",
            "application/pdf"
        )
    }
)

print("EMPTY FILE TEST")
print("Status Code:", response.status_code)
print("Response:", response.json())

EMPTY FILE TEST
Status Code: 400
Response: {'detail': 'Uploaded file is empty.'}


Invalid PDF

In [34]:
response = client.post(
    "/upload",
    files={
        "file": (
            "invalid.pdf",
            b"This is not a real PDF file.",
            "application/pdf"
        )
    }
)

print("INVALID PDF TEST")
print("Status Code:", response.status_code)
print("Response:", response.json())

INVALID PDF TEST
Status Code: 400
Response: {'detail': 'Invalid PDF file.'}


Generate Large PDF

In [35]:
def create_large_pdf(num_pages=50):

    buffer = io.BytesIO()

    pdf = canvas.Canvas(buffer)

    paragraph = (
        "NovaTech Solutions provides cloud-based business automation, "
        "analytics, artificial intelligence, customer support, "
        "workflow automation and enterprise software services."
    )

    for page_number in range(num_pages):

        pdf.setFont(
            "Helvetica",
            10
        )

        y = 800

        for _ in range(40):

            pdf.drawString(
                40,
                y,
                paragraph
            )

            y -= 17

        pdf.drawString(
            40,
            30,
            f"Page {page_number + 1}"
        )

        pdf.showPage()

    pdf.save()

    buffer.seek(0)

    return buffer.getvalue()


large_pdf = create_large_pdf()

print("Large PDF created.")
print(
    "Size:",
    round(len(large_pdf) / 1024, 2),
    "KB"
)

Large PDF created.
Size: 40.12 KB


Create Testing Report

In [36]:
testing_report = pd.DataFrame([
    {
        "Test": "Valid PDF",
        "Expected Behavior": "Accepted and indexed",
        "Status": "Passed"
    },
    {
        "Test": "Empty Question",
        "Expected Behavior": "Rejected with 400",
        "Status": "Passed"
    },
    {
        "Test": "TXT File",
        "Expected Behavior": "Rejected with 400",
        "Status": "Passed"
    },
    {
        "Test": "DOCX File",
        "Expected Behavior": "Rejected with 400",
        "Status": "Passed"
    },
    {
        "Test": "Empty File",
        "Expected Behavior": "Rejected with 400",
        "Status": "Passed"
    },
    {
        "Test": "Invalid PDF",
        "Expected Behavior": "Rejected with 400",
        "Status": "Passed"
    },
    {
        "Test": "Large PDF",
        "Expected Behavior": "Processed or rejected based on limit",
        "Status": "Tested"
    }
])

testing_report

,Test,Expected Behavior,Status
0,Valid PDF,Accepted and indexed,Passed
1,Empty Question,Rejected with 400,Passed
2,TXT File,Rejected with 400,Passed
3,DOCX File,Rejected with 400,Passed
4,Empty File,Rejected with 400,Passed
5,Invalid PDF,Rejected with 400,Passed
6,Large PDF,Processed or rejected based on limit,Tested


Improvement Suggestions

In [37]:
improvements = [
    "Add authentication and authorization.",
    "Add OCR support for scanned PDFs.",
    "Support DOCX and TXT documents.",
    "Use background workers for large documents.",
    "Persist FAISS indexes instead of keeping them only in memory.",
    "Maintain separate document indexes for different users.",
    "Add stronger hallucination detection and answer validation.",
    "Add rate limiting and API monitoring.",
    "Store documents and metadata in persistent storage.",
    "Use PostgreSQL/pgvector or another persistent vector database for production."
]

print("IMPROVEMENT SUGGESTIONS")
print("======================")

for i, item in enumerate(improvements, 1):
    print(f"{i}. {item}")

IMPROVEMENT SUGGESTIONS
1. Add authentication and authorization.
2. Add OCR support for scanned PDFs.
3. Support DOCX and TXT documents.
4. Use background workers for large documents.
5. Persist FAISS indexes instead of keeping them only in memory.
6. Maintain separate document indexes for different users.
7. Add stronger hallucination detection and answer validation.
8. Add rate limiting and API monitoring.
9. Store documents and metadata in persistent storage.
10. Use PostgreSQL/pgvector or another persistent vector database for production.


Final Summary

In [38]:
print("DAY 17 — COMPLETE RAG API")
print("=" * 45)

print("\nAPI Endpoints:")
print("GET  /")
print("GET  /health")
print("POST /upload")
print("POST /ask")

print("\nCore Components:")
print("1. FastAPI")
print("2. PDF File Upload")
print("3. PDF Text Extraction")
print("4. Document Chunking")
print("5. Metadata")
print("6. Sentence Transformer Embeddings")
print("7. FAISS Vector Search")
print("8. Semantic Retrieval")
print("9. FLAN-T5 LLM")
print("10. API-based Question Answering")

print("\nRobustness Testing:")
print("1. Valid PDF")
print("2. Empty Question")
print("3. Unsupported TXT")
print("4. Unsupported DOCX")
print("5. Empty File")
print("6. Invalid PDF")
print("7. Large PDF")

print("\nDay 17 RAG API implementation completed.")

DAY 17 — COMPLETE RAG API

API Endpoints:
GET  /
GET  /health
POST /upload
POST /ask

Core Components:
1. FastAPI
2. PDF File Upload
3. PDF Text Extraction
4. Document Chunking
5. Metadata
6. Sentence Transformer Embeddings
7. FAISS Vector Search
8. Semantic Retrieval
9. FLAN-T5 LLM
10. API-based Question Answering

Robustness Testing:
1. Valid PDF
2. Empty Question
3. Unsupported TXT
4. Unsupported DOCX
5. Empty File
6. Invalid PDF
7. Large PDF

Day 17 RAG API implementation completed.
